# Tangential Volume Registration Workflow

This notebook replaces the ad hoc `notebooks/scratch/stitch_tangential_volume.ipynb` workflow for the **volume-registration part**.

It covers:
- preprocessing prerequisites that the old notebook relied on,
- overview generation across chambers,
- export of a single unregistered overview stack for manual masking,
- automatic pairwise slice registration,
- interactive review of adjacent slice registrations,
- composition of per-slice transforms into a global frame,
- construction of a registered volume stack (parallel + optional isotropic XY),
- optional Z interpolation to fully isotropic voxels,
- optional registration of **already detected / already merged** spot tables into the global frame.

It intentionally does **not** do spot calling. That should happen outside this notebook.

Important differences vs the old scratch notebook:
- `iss export-unregistered-volume` does **not** generate overviews. It collects existing `register_to_ara/*.ome.tif` files.
- Overview generation still comes from the existing preprocessing stack: averages, within-acquisition registration, and `overview_for_ara_registration(...)`.
- `build_registered_volume_stack(...)` uses the overview metadata to choose the XY voxel size, locally averages each stitched slice down to that voxel size, then warps at the smaller canvas; `n_jobs` dispatches per-slice work via joblib workers.

In [ ]:

from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from scipy.ndimage import zoom
from skimage.transform import AffineTransform, warp

import iss_preprocess as iss
from iss_preprocess.io import get_pixel_size, get_processed_path, load_metadata
from iss_preprocess.pipeline.pipeline import average, register_acquisition, overview_for_ara_registration
from iss_preprocess.pipeline.stitch import register_all_rois_within
from iss_preprocess.pipeline.volume_registration import (
    apply_affine_to_points,
    build_registered_volume_stack,
    compose_global_slice_transforms,
    export_unregistered_volume_stack,
    get_volume_root,
    load_global_slice_transforms,
    load_unregistered_volume_stack,
    reduce_pairwise_state,
    register_adjacent_slices,
    register_single_pair,
    register_somata_to_global_volume,
    register_spots_to_global_volume,
    register_tissue_masks_to_global_volume,
    warp_with_affine,
)
from iss_preprocess.vis import review_pairwise_registrations_widget

%load_ext autoreload
%autoreload 2

## Parameters

Set these before running anything else.

In [ ]:

mouse_path = "essenbd_projdev/BRAC11398.3c"
chambers = ["01", "02", "03", "04"]
ref_acq = "hybridisation_round_1_1"   # acquisition used to build overview slices
volume_prefix = "hybridisation_round_1_1"  # acquisition to warp into the final volume
spots_prefix = "barcode_round"  # optional, only if spot tables already exist
barcode_ref_round = "barcode_round_2_1"  # optional, only if spot tables already exist

# You can pass either one chamber path or the mouse path below.
# The volume-registration code will gather all sibling chambers under the same mouse folder.
path_for_volume_stage = mouse_path

data_paths = [f"{mouse_path}/chamber_{ch}" for ch in chambers]


## 0. Notebook-Extraction Rule

For reference, I extracted the old scratch notebook into a linear script before mapping the workflow:
- `notebooks/scratch/stitch_tangential_volume_extracted.py`

That file is useful when you want to compare the old notebook's true execution order against the new structured workflow below.

## 1. Preprocessing Prerequisites

These are the steps the old scratch notebook ran **before** volume registration:
1. averages for illumination correction,
2. channel / round registration inside the acquisition,
3. within-ROI tile registration,
4. ROI overview generation into `processed/.../register_to_ara/*.ome.tif`.

The new volume-registration code starts **after** these overview files exist.

In [ ]:

def find_rois_for_prefix(data_path, prefix):
    processed_path = get_processed_path(data_path)
    roi_pattern = re.compile(r"(\d+)-Pos\d+_")
    roi_files = sorted((processed_path / prefix).glob("*-Pos*_*.tif"))
    return sorted({int(m.group(1)) for f in roi_files if (m := roi_pattern.search(f.name))})


In [ ]:

# 1a. Averages used for illumination correction
# Uncomment to run.
# for data_path in data_paths:
#     average(data_path)


In [ ]:

# 1b. Register channels / rounds inside the reference acquisition
# Uncomment to run.
# for data_path in data_paths:
#     register_acquisition(data_path=data_path, prefix=ref_acq, force_redo=False)


In [ ]:

# 1c. Register tiles within each ROI for the reference acquisition
# Uncomment to run.
# for data_path in data_paths:
#     rois = find_rois_for_prefix(data_path, ref_acq)
#     register_all_rois_within(
#         data_path,
#         prefix=ref_acq,
#         ref_ch=1,
#         suffix="max",
#         correct_illumination=False,
#         roi2use=rois,
#         reload=False,
#         save_plot=True,
#         dimension_prefix=ref_acq,
#         use_slurm=True,
#         scripts_name=f"register_within_{ref_acq}_{Path(data_path).name}",
#     )


In [ ]:

# 1d. Generate stitched overview files in processed/.../register_to_ara/
# This is the step the new volume stage expects to have been done already.
# Uncomment to run.
# for data_path in data_paths:
#     rois = find_rois_for_prefix(data_path, ref_acq)
#     overview_for_ara_registration(
#         data_path=data_path,
#         prefix=ref_acq,
#         rois_to_do=rois,
#         sigma_blur=1,
#         ref_prefix=ref_acq,
#     )


## 2. Export One Unregistered Stack for Manual Masking

This replaces the old notebook's manual "find largest slice / pad all slices / draw masks" setup.

What it does:
- scans all overview files in `register_to_ara` across chambers,
- sorts them by `slice_number`,
- center-pads them to a common canvas,
- writes one compressed `.npz` with:
  - `images`,
  - `default_masks`,
  - `user_masks`,
  - `manifest_json`.

What it does **not** do:
- it does not generate the overview files itself.

In [ ]:

stack_path = export_unregistered_volume_stack(path_for_volume_stage, overwrite=False)
stack_path


In [ ]:
stack_path = Path("/nemo/project/proj-znamenp-barseq/processed/essenbd_projdev/BRAC11398.3c/tangential_volume/unregistered_slices.npz")
stack_path

In [ ]:

stack = load_unregistered_volume_stack(stack_path)
manifest = stack["manifest"]
entries = manifest["entries"]
print(f"Number of slices: {len(entries)}")
print(f"Common canvas shape: {tuple(manifest['target_shape_yx'])}")
print(pd.DataFrame(entries)[["data_path", "roi", "slice_number", "overview_shape_yx"]].head())


### Manual Masking Step

At this point:
1. copy `stack_path` from the VM to a local temp folder,
2. open the local `.npz` in Napari,
3. add `images` as an image layer and `user_masks` as a labels layer,
4. edit `user_masks`,
5. save the edited `user_masks` back into the same local `.npz`,
6. upload the edited `.npz` back to the same `stack_path` on the VM.

Run these lines in the Napari console, one block at a time:

```python
from pathlib import Path
import numpy as np

stack_path = Path("/path/to/local/tmp/unregistered_slices.npz")
data = np.load(stack_path, allow_pickle=False)
```

```python
images = data["images"]
default_masks = data["default_masks"].astype(np.uint8)
manifest_json = str(data["manifest_json"].item())

# Start from a fresh blank mask. Recommended default.
# If user_masks already contains 1s from a previous save, it will look
# "all filled" in napari — that is the stored data, not a display bug.
user_masks = np.zeros_like(default_masks, dtype=np.uint8)

# To resume a previous drawing session instead, replace the line above with:
# user_masks = data["user_masks"].astype(np.uint8)
```

```python
viewer.add_image(images, name="unregistered_slices")
```

```python
mask_layer = viewer.add_labels(user_masks, name="user_masks", opacity=0.35)
```

Edit the `user_masks` layer with Napari's paint and erase tools:
- `1` = keep for registration
- `0` = exclude from registration

```python
np.savez_compressed(
    stack_path,
    images=images,
    default_masks=default_masks,
    user_masks=(mask_layer.data > 0).astype(np.uint8),
    manifest_json=manifest_json,
)
```

After re-uploading the edited file to the VM, the pairwise registration step below will use `user_masks` instead of raw rectangular overlap.

## 3. Automatic Pairwise Slice Registration

This replaces the old notebook's manual loop over adjacent slice pairs.

What it covers from the old notebook:
- adjacent pair creation,
- fixed / moving ordering,
- center-padding to a common canvas,
- orientation candidates,
- rotation estimate,
- masked phase correlation,
- saved per-pair transform state.

In [ ]:
# Serial (default): register every pair in this process.
state_path = register_adjacent_slices(
    stack_path=stack_path,
    state_path=None,
    force=False,
    local_search_radius=None,
    angle_radius_deg=5.0,
    rotation_hann=True,
    apply_translation_hann=True,
    background_mode="percentile_subtract",
    background_percentile=5.0,
    clip_percentile=99.5,
    normalize_mode="contrast_stretch",  # or "clahe", "none"
)
state_path

# Parallel (SLURM): one job per pair + a reducer. Returns job ids.
# Curation flow: after the widget marks overrides / clears 'accepted',
# resubmit with use_slurm=True to recompute just the non-accepted pairs.
# slurm_result = register_adjacent_slices(
#     stack_path=stack_path,
#     state_path=None,
#     force=False,
#     local_search_radius=None,
#     angle_radius_deg=5.0,
#     rotation_hann=True,
#     apply_translation_hann=True,
#     background_mode="percentile_subtract",
#     background_percentile=5.0,
#     clip_percentile=99.5,
#     normalize_mode="contrast_stretch",
#     use_slurm=True,
# )
# slurm_result

# Re-register a single pair on SLURM (e.g. after widget curation).
# Then call reduce_pairwise_state(stack_path) to merge it back.
# register_single_pair(
#     stack_path=str(stack_path),
#     fixed_slice=42,
#     moving_slice=41,
#     use_slurm=True,
# )
# reduce_pairwise_state(stack_path=str(stack_path))


In [ ]:
state_path = Path('/nemo/project/proj-znamenp-barseq/processed/essenbd_projdev/BRAC11398.3c/tangential_volume/pairwise_registration_state.json')


## 4. Interactive Review Widget

Two equivalent reviewers are available; pick one. Both read and write the same
`pairwise_registration_state.json`, so you can switch between them freely.

- **`review_pairwise_registrations_napari`** — recommended for routine review
  on the VM. Pops a Qt window via napari; GPU canvas, multiscale layers,
  smooth pan/zoom, instant contrast adjustment. Requires a display server on
  the host (VNC / NoMachine / X11 forwarding) and the `napari` extra:
  `pip install 'iss-preprocess[napari]'`.

- **`review_pairwise_registrations_widget`** — browser-only ipywidgets
  reviewer, no extra deps. Slower on the big stitched slices (rebuilds a
  matplotlib figure and ships a PNG per change) but works in any notebook
  without a display server.

Both reviewers let you go pair by pair and:
- accept the current result,
- force a different orientation / flip,
- mark the fixed or moving slice as bad,
- provide a manual initial `dx / dy / angle`,
- run a **local simulated annealing** refinement around that manual seed.

**Marking bad slices.** When you set the `Bad slice` dropdown to `Fixed` or
`Moving` and click `Accept`, the reviewer:
1. adds that slice to `state['bad_slices']`;
2. auto-registers the new pair that bridges the gap (the next non-bad
   neighbours on each side), so the slider surfaces the freshly needed pair
   with a result already computed;
3. mirrors the updated `bad_slices` into each chamber's `ops.yml` so
   downstream code (e.g. `iss-qc-sindbis`) can read it via
   `iss.io.load_ops(chamber_path)['bad_slices']`.

The source of truth remains `tangential_volume/pairwise_registration_state.json`;
chamber `ops.yml` is a convenience replica.

### Standalone launch (without a notebook)

The same reviewer can be opened from a plain Python prompt on a VM with a
display:

```python
import napari
from iss_preprocess.vis import review_pairwise_registrations_napari

viewer = review_pairwise_registrations_napari(stack_path)
napari.run()  # blocks until you close the window
```


In [ ]:
# Recommended: napari-based reviewer (Qt window, GPU canvas).
# Requires the `napari` extra and a display server (VNC / NoMachine / X11) on the host.
from iss_preprocess.vis import review_pairwise_registrations_napari

viewer = review_pairwise_registrations_napari(stack_path, state_path=state_path)
viewer

# Fallback: in-notebook ipywidgets reviewer (slower on big slices, no extra deps).
# from iss_preprocess.vis import review_pairwise_registrations_widget
# review_pairwise_registrations_widget(stack_path, state_path=state_path)


## 5. Compose Global Slice Transforms

This replaces the old notebook's anchor-slice selection and transform concatenation.

In [ ]:

transforms_path = compose_global_slice_transforms(
    stack_path=stack_path,
    state_path=state_path,
    reference_slice=None,          # or set an explicit slice number
    reference_strategy="largest_area",
)
transforms_path


In [ ]:
transforms_path = Path('/nemo/project/proj-znamenp-barseq/processed/essenbd_projdev/BRAC11398.3c/tangential_volume/global_slice_transforms.npz')

In [ ]:

transforms = load_global_slice_transforms(transforms_path)
print("Reference slice:", transforms["reference_slice"])
print("Canvas shape:", transforms["canvas_shape_yx"])
print(pd.DataFrame({
    "slice_number": transforms["slice_numbers"],
    "roi": transforms["rois"],
    "data_path": transforms["data_paths"],
}).head())


## 6. Quick QC: Warp a Few Overview Slices into the Global Frame

This reproduces the old notebook's "show that I can warp any slice to the global frame" sanity check.

In [ ]:

transforms = load_global_slice_transforms(transforms_path)
show_n = min(4, len(transforms["slice_numbers"]))
fig, axes = plt.subplots(1, show_n, figsize=(5 * show_n, 5))
if show_n == 1:
    axes = [axes]
for ax, overview_file, slice_number, matrix in zip(
    axes,
    transforms["overview_files"][:show_n],
    transforms["slice_numbers"][:show_n],
    transforms["global_from_overview"][:show_n],
):
    image = tifffile.imread(overview_file).astype(np.float32)
    warped = warp_with_affine(image, matrix, output_shape=transforms["canvas_shape_yx"], order=1)
    ax.imshow(warped, cmap="gray")
    ax.set_title(f"slice {slice_number}")
    ax.axis("off")
plt.tight_layout()


## 7. Build the Registered Volume Stack

Stitches each ROI for `volume_prefix`, warps it into the global frame, and stores
the slices in a registered stack indexed by physical slice spacing.

Two important kwargs:

- `target_voxel_size_um` — when set, each stitched channel is downsampled with
  local-mean/anti-aliased resampling to that XY voxel size before the affine
  warp. The output canvas is sized from the overview pixel size recorded in
  `register_to_ara/*.ome.yml`, matching the old scratch workflow's
  `downsample_ratio * original_pixel_size / voxel_size` accounting. Leave at
  `None` to keep the registration-overview pixel grid.
- `n_jobs` — `1` runs sequentially (default). `>1` dispatches per-slice work via
  joblib `loky` workers with BLAS oversubscription guarded by
  `threadpool_limits(limits=1)`. Throughput is typically disk-bound, so values
  much above the number of independent disk readers see diminishing returns.

In [ ]:
target_voxel_size_um = 10.0  # set to None to keep the native stitched-tile pixel grid
n_jobs = 10                    # set to 1 for the sequential path

volume_path = build_registered_volume_stack(
    data_path=path_for_volume_stage,
    transforms_path=transforms_path,
    prefix=volume_prefix,
    output_name=None,
    suffix="max",
    z_step_um=None,
    target_voxel_size_um=target_voxel_size_um,
    n_jobs=n_jobs,
)
volume_path


In [ ]:
volume_path = Path("/nemo/project/proj-znamenp-barseq/processed/essenbd_projdev/BRAC11398.3c/tangential_volume/registered_volume_hybridisation_round_1_1_iso10um.npz")

In [ ]:
volume_npz = np.load(volume_path)
print("volume shape (z, y, x, ch):", volume_npz["volume"].shape)
print("xy voxel size (um):", float(volume_npz["xy_voxel_size_um"]))
print("z step (um):", float(volume_npz["z_step_um"]))
print("slice numbers:", volume_npz["slice_numbers"][:10])
print("z positions (um):", volume_npz["z_positions_um"][:10])


## 8. Optional: Z Interpolation to Fully Isotropic Voxels

If you set `target_voxel_size_um` in section 7, **XY is already isotropic**. The
remaining anisotropy is along Z (physical slice spacing vs. target voxel size).
This section linearly interpolates Z to the same isotropic voxel size if you
need a fully isotropic 3D volume (e.g. for atlas alignment or 3D rendering).

If you ran section 7 with `target_voxel_size_um=None`, you'll need to do XY
downsampling here too — see the commented-out branch in the next code cell.

Notes:
- channels are kept separate and are **not** mixed during interpolation,
- this is a notebook-side postprocessing step.

In [ ]:
isotropic_output_name = None  # default: derive from native volume file name

native_volume = volume_npz["volume"].astype(np.float32)
native_z_step_um = float(volume_npz["z_step_um"])
xy_voxel_size_um = float(volume_npz["xy_voxel_size_um"])
# Target the same voxel size in Z as the volume already has in XY.
target_voxel_size_um = xy_voxel_size_um

z_scale = native_z_step_um / target_voxel_size_um
# If XY isn't already isotropic (i.e. you ran section 7 with target_voxel_size_um=None),
# uncomment and set xy_scale to xy_voxel_size_um / target_voxel_size_um and pass it below.
xy_scale = 1.0

print(f"Native volume shape: {native_volume.shape}")
print(f"XY voxel size (um): {xy_voxel_size_um:.4f}")
print(f"Native Z step (um): {native_z_step_um:.4f}")
print(f"Target isotropic voxel size (um): {target_voxel_size_um:.4f}")
print(f"XY scale factor: {xy_scale:.4f}")
print(f"Z scale factor: {z_scale:.4f}")


In [ ]:
# volume shape is (z, y, x, channel); channel axis kept at 1.0 so it's untouched.
isotropic_volume = zoom(
    native_volume,
    zoom=(z_scale, xy_scale, xy_scale, 1.0),
    order=1,
)

print(f"Isotropic volume shape: {isotropic_volume.shape}")


In [ ]:

# quick preview: middle z plane, first channel
mid_z_native = native_volume.shape[0] // 2
mid_z_iso = isotropic_volume.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(native_volume[mid_z_native, :, :, 0], cmap="gray")
axes[0].set_title(f"Native volume, z={mid_z_native}, ch=0")
axes[1].imshow(isotropic_volume[mid_z_iso, :, :, 0], cmap="gray")
axes[1].set_title(f"Isotropic volume, z={mid_z_iso}, ch=0")
for ax in axes:
    ax.axis("off")
plt.tight_layout()


In [ ]:
# save_isotropic_volume
isotropic_path = Path(volume_path)
if isotropic_output_name is None:
    isotropic_output_name = (
        isotropic_path.stem + f"_isotropic_{int(target_voxel_size_um)}um.ome.tif"
    )
isotropic_path = isotropic_path.with_name(isotropic_output_name)
# OME-TIFF / Fiji hyperstack convention: (Z, C, Y, X). Our isotropic_volume
# is (Z, Y, X, C); the transpose is a view, tifffile copies per page.
tifffile.imwrite(
    isotropic_path,
    np.transpose(isotropic_volume, (0, 3, 1, 2)),
    photometric="minisblack",
    bigtiff=True,
    ome=True,
    metadata={
        "axes": "ZCYX",
        "PhysicalSizeX": float(target_voxel_size_um),
        "PhysicalSizeXUnit": "µm",
        "PhysicalSizeY": float(target_voxel_size_um),
        "PhysicalSizeYUnit": "µm",
        "PhysicalSizeZ": float(target_voxel_size_um),
        "PhysicalSizeZUnit": "µm",
    },
)
print(f"Saved isotropic OME-TIFF to {isotropic_path}")


## 9. Optional: Register Existing Spots to the Global Frame

This is the replacement for the **global spot registration** part of the old notebook.

It assumes the spots already exist and are already merged within each ROI, for example via:
- `merge_and_align_spots_all_rois(...)`, or
- whatever ROI-level spot table generation you use outside this notebook.

This step does **not** do spot calling.

In [ ]:
from iss_preprocess.pipeline.align_spots_and_cells import merge_and_align_spots_all_rois

for data_path in data_paths:
    merge_and_align_spots_all_rois(data_path, spots_prefix, barcode_ref_round)

In [ ]:
# Run after ROI-level spot tables like processed/.../{spots_prefix}_spots_{roi}.pkl have been generated.
# `output_unit="um"` makes downstream QC code interpretable in physical units (μm).
global_spots_path = register_spots_to_global_volume(
    data_path=path_for_volume_stage,
    transforms_path=transforms_path,
    spots_prefix=spots_prefix,
    pixel_size_reference_round="barcode_round_1_1",
    output_name=None,
    output_unit="um",
)
global_spots_path


## 9b. Register Stitched Soma Calls to the Global Frame

Same per-slice `global_from_fullres` affine, applied to the per-chamber
stitched soma table from
`iss_preprocess.pipeline.somata.load_stitched_soma_calls(..., filtered=False)`.
Outputs `tangential_volume/{barcode_prefix}_somata_global.pkl` for
consumption by the downstream QC pipeline (iss-qc-sindbis).

`filtered=False` is intentional: soma QC thresholds belong in the QC
pipeline, not here.


In [ ]:
global_somata_path = register_somata_to_global_volume(
    data_path=path_for_volume_stage,
    transforms_path=transforms_path,
    barcode_prefix=spots_prefix,
    pixel_size_reference_round="barcode_round_1_1",
    output_name=None,
    output_unit="um",
)
global_somata_path


## 9c. Warp User-Drawn Tissue Masks Into the Global Frame

Reads `user_masks` from `unregistered_slices.npz` (drawn in step 2 above),
applies `canvas_offset_matrix @ global_from_padded[i]` per slice via
nearest-neighbour warp, and writes a 3D boolean volume in the same canvas
as `build_registered_volume_stack` (native overview resolution, with
metadata for μm conversion).

ROIs with no mask drawn are auto-flagged as bad slices upstream by
`_prepare_pairwise_state` and are not in `transforms["slice_numbers"]`, so
they're naturally excluded.

Output: `tangential_volume/tissue_mask_global.npz` — consumed by
`iss-qc-sindbis` core-qc.


In [ ]:
global_tissue_mask_path = register_tissue_masks_to_global_volume(
    data_path=path_for_volume_stage,
    transforms_path=transforms_path,
    stack_path=stack_path,
    output_name="tissue_mask_global.npz",
    z_step_um=None,
)
global_tissue_mask_path


## 10. Coverage Map: Old Scratch Notebook vs New Workflow

### Covered directly by the new structured workflow
- overview discovery across chambers,
- largest-canvas / center-padding logic,
- export of one unified unregistered slice stack,
- mask-aware adjacent pair registration,
- orientation / flip choice,
- bad-slice exclusion,
- manual seeded local optimisation with simulated annealing,
- concatenation of per-pair transforms into global slice transforms,
- warped overview QC,
- registered ROI-stack assembly,
- separate global spot-registration stage.

### Still handled by existing preprocessing functions, not by the new volume module itself
- averages for illumination correction,
- round / channel registration inside the acquisition,
- within-ROI tile registration,
- overview generation into `register_to_ara/*.ome.tif`.

### Not reproduced yet from the old scratch notebook
- blood-vessel-specific overview side paths,
- ad hoc plotting branches that were only exploratory.